# openspin.noise — interactive demo

Fast correlated (temporal & spatial) charge/qubit noise generator.

**Contents:**
1. Generate 1/f noise by name
2. OU-sum (physically-motivated 1/f)
3. From a measured trace (calibrate + resynthesize)
4. Analysis: PSD, Allan deviation, ACF
5. Random telegraph noise (RTN)
6. Spatial noise
7. Composition: drift + quasistatic + white
8. Backend comparison (numpy vs numba vs jax)

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from openspin.noise import (
    generate, calibrate, list_processes,
    welch, allan_deviation, autocorrelation,
    compose, add_drift, add_quasistatic, add_white,
)

print("processes:", list_processes())

ModuleNotFoundError: No module named 'openspin'

## 1. Generate 1/f noise by name

Timmer-König FFT method. Stationary Gaussian 1/f^alpha.

In [ ]:
res = generate(
    "1/f",
    n_traj=200,
    fs=1e4,
    n_points=2**14,
    alpha=1.0,
    S0=1.0, f0=1.0,
    seed=0,
)
print(f"traj shape: {res.traj.shape}")
print(f"time grid: {res.t.shape}")
print(f"fs: {res.fs} Hz")
print(f"units: {res.units}")
print(f"seed: {res.seed}")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(res.t[:1000], res.traj[0, :1000], lw=0.8)
ax1.set(xlabel="Time [s]", ylabel="Noise", title="Single trajectory")

f, S = welch(res.traj, fs=res.fs)
ax2.loglog(f[1:], S[1:], label="measured")
ax2.loglog(f[1:], 1.0/f[1:], "k--", alpha=0.5, label="target 1/f")
ax2.set(xlabel="Frequency [Hz]", ylabel="PSD [units²/Hz]", title="Power spectrum")
ax2.legend()

fig.tight_layout()

## 2. OU-sum (physically-motivated 1/f)

Sum of Ornstein-Uhlenbeck components. Each component is a Lorentzian; sum approximates 1/f^alpha.

In [ ]:
res_ou = generate(
    "ou_sum",
    n_traj=100,
    fs=1e4,
    n_points=2**14,
    alpha=1.0,
    S0=1.0, f0=1.0,
    n_components_per_decade=8,
    seed=0,
    backend="numpy",
)
print(f"traj shape: {res_ou.traj.shape}")
print(f"spec keys: {list(res_ou.spec.keys())}")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(res_ou.t[:1000], res_ou.traj[0, :1000], lw=0.8)
ax1.set(xlabel="Time [s]", ylabel="Noise", title="OU-sum trajectory")

f_ou, S_ou = welch(res_ou.traj, fs=res_ou.fs)
ax2.loglog(f_ou[1:], S_ou[1:], label="OU-sum")
ax2.loglog(f_ou[1:], 1.0/f_ou[1:], "k--", alpha=0.5, label="target 1/f")
ax2.set(xlabel="Frequency [Hz]", ylabel="PSD [units²/Hz]", title="OU-sum spectrum")
ax2.legend()

fig.tight_layout()

## 3. From a measured trace (calibrate + resynthesize)

Estimate second-order statistics from a trace, then generate surrogate trajectories.

In [ ]:
# synthesize a fake "measured" trace
trace = generate("1/f", n_traj=1, fs=1e4, n_points=2**12,
                 alpha=1.0, S0=1.0, f0=1.0, seed=42).traj[0]

gen = calibrate(trace, fs=1e4, method="circulant")
surr = gen.sample(n_traj=100, n_points=2**12, seed=0)

print(f"original trace: {trace.shape}")
print(f"surrogate traj: {surr.traj.shape}")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(surr.t, trace, label="original", alpha=0.7)
ax1.plot(surr.t, surr.traj[0], label="surrogate", alpha=0.7)
ax1.set(xlabel="Time [s]", ylabel="Noise", title="Original vs surrogate")
ax1.legend()

f_t, S_t = welch(trace, fs=surr.fs)
f_s, S_s = welch(surr.traj, fs=surr.fs)
ax2.loglog(f_t[1:], S_t[1:], label="original")
ax2.loglog(f_s[1:], S_s[1:], label="surrogate (mean)", alpha=0.8)
ax2.set(xlabel="Frequency [Hz]", ylabel="PSD [units²/Hz]", title="PSD match")
ax2.legend()

fig.tight_layout()

## 4. Analysis: PSD, Allan deviation, ACF

In [ ]:
res = generate("1/f", n_traj=100, fs=1e4, n_points=2**14,
               alpha=1.0, S0=1.0, f0=1.0, seed=0)

f, S = welch(res.traj, fs=res.fs)
taus, sigma = allan_deviation(res.traj, fs=res.fs)
lags, acf_vals = autocorrelation(res.traj, unbiased=True)

fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(14, 4))

ax1.loglog(f[1:], S[1:])
ax1.set(xlabel="Frequency [Hz]", ylabel="PSD [units²/Hz]", title="Power spectrum")

ax2.loglog(taus, sigma)
ax2.set(xlabel="Averaging time τ [s]", ylabel="Allan deviation σ(τ)", title="Allan deviation")

ax3.plot(lags[:200], acf_vals[:200])
ax3.set(xlabel="Lag [samples]", ylabel="ACF", title="Autocorrelation")

fig.tight_layout()

## 5. Random telegraph noise (RTN)

Two-level fluctuators with Poisson switching.

In [ ]:
res_rtn = generate(
    "rtn",
    n_traj=10,
    fs=1e4,
    n_points=2**12,
    tau_up=1e-3, tau_down=1e-3,
    amplitude=1.0,
    seed=0,
)

fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(res_rtn.t[:2000], res_rtn.traj[0, :2000], drawstyle="steps-post", lw=1)
ax.set(xlabel="Time [s]", ylabel="Amplitude", title="RTN trajectory")
fig.tight_layout()

## 6. Spatial noise

Spatially correlated noise via a kernel (e.g. exponential decay).

In [ ]:
res_sp = generate(
    "spatial_fluctuators",
    n_traj=1,
    n_sites=64,
    fs=1e4,
    n_points=2**10,
    correlation_length=10.0,
    amplitude=1.0,
    seed=0,
)
print(f"traj shape: {res_sp.traj.shape}  (n_traj, n_sites, n_time)")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

im = ax1.imshow(res_sp.traj[0], aspect="auto", cmap="RdBu_r",
                extent=[0, res_sp.t[-1], 0, res_sp.n_sites])
ax1.set(xlabel="Time [s]", ylabel="Site", title="Spatial noise (space-time)")
plt.colorbar(im, ax=ax1)

ax2.plot(res_sp.t, res_sp.traj[0, 0], label="site 0", lw=0.8)
ax2.plot(res_sp.t, res_sp.traj[0, 16], label="site 16", lw=0.8)
ax2.plot(res_sp.t, res_sp.traj[0, 32], label="site 32", lw=0.8)
ax2.set(xlabel="Time [s]", ylabel="Noise", title="Selected sites")
ax2.legend()

fig.tight_layout()

## 7. Composition: drift + quasistatic + white

Build complex noise from simple building blocks.

In [ ]:
t = np.linspace(0, 1, 2**12)
fs = 1.0 / (t[1] - t[0])

res = generate("1/f", n_traj=10, t=t, alpha=1.0, S0=1.0, f0=1.0, seed=0)

composed = compose(
    res.traj,
    add_drift(rate=0.5),
    add_quasistatic(offset=2.0, sigma=0.3, seed=1),
    add_white(sigma=0.1, seed=2),
)

fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(t, composed[0], lw=0.8)
ax.set(xlabel="Time [s]", ylabel="Noise", title="1/f + drift + quasistatic + white")
fig.tight_layout()

## 8. Backend comparison (numpy vs numba vs jax)

OU-family generators support multiple backends. Compare wall time.

In [ ]:
import time

backends = []
for b in ("numpy", "numba", "jax"):
    try:
        t0 = time.perf_counter()
        _ = generate("ou", n_traj=100, fs=1e4, n_points=2**14,
                     theta=100.0, mu=0.0, sigma=1.0, seed=0, backend=b)
        dt = time.perf_counter() - t0
        backends.append((b, dt))
        print(f"{b:8s}: {dt:.3f} s")
    except Exception as e:
        print(f"{b:8s}: unavailable ({e})")

if backends:
    fig, ax = plt.subplots(figsize=(6, 3))
    names, times = zip(*backends)
    ax.bar(names, times)
    ax.set(ylabel="Wall time [s]", title="OU backend comparison")
    fig.tight_layout()

## Summary

| Feature | API |
|---------|-----|
| Generate by name | `generate("1/f", ...)` |
| Generate from trace | `calibrate(trace, fs).sample(...)` |
| PSD | `welch(traj, fs)` |
| Allan deviation | `allan_deviation(traj, fs)` |
| ACF | `autocorrelation(traj)` |
| Compose | `compose(traj, add_drift(...), add_white(...))` |
| Backends | `backend="numpy"` / `"numba"` / `"jax"` |
| Save/load | `res.save(path)` / `NoiseResult.load(path)` |

See `README.md` and `spec.md` for full API reference.